# VideoTransDub - Dich & Long Tieng Video Tren Colab

## Cach su dung (3 buoc)

1. **Chon GPU**: Menu `Runtime` > `Change runtime type` > chon **T4 GPU**
2. **Chay tat ca**: Menu `Runtime` > `Run all` (hoac Ctrl+F9)
3. **Doi link**: Cell 4 se hien **1 URL mau xanh** (dang `https://xxx.trycloudflare.com`)
   - Click vao URL do = mo giao dien web de upload video, chon ngon ngu, xem ket qua
   - Giao dien chay tren trinh duyet, khong can cai gi them

### Luu y
- Toan bo qua trinh cai dat mat khoang **3-5 phut** lan dau
- URL thay doi moi lan mo Colab, nhung **video dang xu ly khong bi mat** (luu tren Google Drive)
- Neu bi ngat ket noi, chay lai Cell 4 de lay URL moi -- pipeline tu dong tiep tuc tu cho cu

---

## Cell 1: Mount Google Drive & Clone Du An

> Cell nay tu dong mount Drive va clone repo. **Ban chi can sua 1 dong**: URL repo GitHub cua ban.

In [ ]:
# ============================================================
# Cell 1: Google Drive + Clone Repository
# ============================================================
#
# ⚠️ SUA DONG DUOI DAY thanh URL repo GitHub cua ban:
GITHUB_REPO = "https://github.com/lytuan123/videotrans.git"
#
# ============================================================

import os
from pathlib import Path

# --- 1. Mount Google Drive ---
GDRIVE_ROOT = Path('/content/drive')
GDRIVE_VTD = Path('/content/drive/MyDrive/VideoTransDub')

if not GDRIVE_ROOT.exists():
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)

GDRIVE_VTD.mkdir(parents=True, exist_ok=True)
(GDRIVE_VTD / 'output').mkdir(exist_ok=True)
(GDRIVE_VTD / 'checkpoints').mkdir(exist_ok=True)
print(f'✅ Google Drive mounted -> {GDRIVE_VTD}')

# --- 2. Clone / update repo ---
REPO_DIR = Path('/content/pyvideotrans')
if not REPO_DIR.exists():
    !git clone {GITHUB_REPO} /content/pyvideotrans
    print('✅ Repository cloned')
else:
    %cd /content/pyvideotrans
    !git pull --ff-only 2>/dev/null || echo '⚠️ Pull skipped (co thay doi local)'
    print('✅ Repository updated')

%cd /content/pyvideotrans
os.chdir('/content/pyvideotrans')  # Dam bao CWD cho subprocess
print(f'📂 Working directory: {os.getcwd()}')

## Cell 2: Cai Dat (tu dong, khong can lam gi)

> Cell nay cai ffmpeg, Whisper, Edge-TTS, Streamlit, Cloudflare tunnel... Mat 3-5 phut.

In [ ]:
# ============================================================
# Cell 2: Environment Setup - Force Install Everything
# ============================================================
import subprocess, shutil, sys

def setup_environment():
    """No-fail environment setup for Colab."""
    errors = []

    # --- 1. System packages ---
    print('📦 Installing system packages...')
    subprocess.run(['apt-get', 'update', '-qq'], capture_output=True)
    result = subprocess.run(
        ['apt-get', 'install', '-y', '-qq', 'ffmpeg', 'libsndfile1'],
        capture_output=True, text=True
    )
    if result.returncode != 0:
        errors.append(f'apt-get failed: {result.stderr}')

    # Verify ffmpeg
    ffmpeg_path = shutil.which('ffmpeg')
    if ffmpeg_path:
        ver = subprocess.run(['ffmpeg', '-version'], capture_output=True, text=True)
        print(f'  ✅ ffmpeg: {ver.stdout.splitlines()[0]}')
    else:
        errors.append('ffmpeg not found after install!')
        print('  ❌ ffmpeg NOT FOUND')

    ffprobe_path = shutil.which('ffprobe')
    if ffprobe_path:
        print(f'  ✅ ffprobe: found at {ffprobe_path}')
    else:
        print('  ⚠️ ffprobe not found')

    # --- 2. Python packages ---
    print('\n🐍 Installing Python packages...')
    packages = [
        'pydantic>=2,<3',
        'PyYAML>=6,<7',
        'faster-whisper>=1.0.0',
        'edge-tts>=6.1.0',
        'opencv-python-headless>=4.8.0',
        'streamlit>=1.30.0',
        'psutil>=5.9.0',
        'ipywidgets>=8.0.0',
        'dashscope>=1.20.0',
    ]
    for pkg in packages:
        name = pkg.split('>=')[0].split('<')[0]
        result = subprocess.run(
            [sys.executable, '-m', 'pip', 'install', '--quiet', pkg],
            capture_output=True, text=True
        )
        if result.returncode == 0:
            print(f'  ✅ {name}')
        else:
            errors.append(f'pip install {name} failed: {result.stderr}')
            print(f'  ❌ {name}: {result.stderr[:100]}')

    # --- 3. Install videotransdub package ---
    print('\n📦 Installing videotransdub...')
    result = subprocess.run(
        [sys.executable, '-m', 'pip', 'install', '--quiet', '-e',
         'apps/videotransdub'],
        capture_output=True, text=True
    )
    if result.returncode == 0:
        print('  ✅ videotransdub installed')
    else:
        errors.append(f'videotransdub install failed: {result.stderr}')
        print(f'  ❌ videotransdub: {result.stderr[:200]}')

    # --- 4. Cloudflare tunnel ---
    print('\n🌐 Installing Cloudflare tunnel...')
    if not shutil.which('cloudflared'):
        subprocess.run(
            ['wget', '-q', 'https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb',
             '-O', '/tmp/cloudflared.deb'],
            capture_output=True
        )
        subprocess.run(['dpkg', '-i', '/tmp/cloudflared.deb'], capture_output=True)
    if shutil.which('cloudflared'):
        print('  ✅ cloudflared ready')
    else:
        print('  ⚠️ cloudflared not available (will use Colab proxy)')

    # --- 5. Runtime dirs ---
    print('\n📂 Creating runtime directories...')
    from pathlib import Path
    for d in ['runtime/workspace', 'runtime/output', 'runtime/uploads']:
        p = Path(d)
        p.mkdir(parents=True, exist_ok=True)
    print('  ✅ Runtime dirs ready')

    # --- 6. GPU check ---
    print('\n🖥️ System info:')
    try:
        gpu = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total',
                              '--format=csv,noheader'], capture_output=True, text=True)
        if gpu.returncode == 0:
            print(f'  ✅ GPU: {gpu.stdout.strip()}')
        else:
            print('  ⚠️ No GPU detected (ASR will use CPU)')
    except FileNotFoundError:
        print('  ⚠️ nvidia-smi not found')

    import psutil
    mem = psutil.virtual_memory()
    print(f'  ✅ RAM: {mem.total / (1024**3):.1f} GB')
    disk = psutil.disk_usage('/')
    print(f'  ✅ Disk: {disk.free / (1024**3):.1f} GB free')

    # --- Summary ---
    if errors:
        print(f'\n⚠️ Setup completed with {len(errors)} warnings:')
        for e in errors:
            print(f'  - {e[:120]}')
    else:
        print('\n✅ Setup complete! All dependencies installed successfully.')

    return len(errors) == 0

setup_ok = setup_environment()

## Cell 3: Kiem Tra He Thong (GPU / RAM / Disk)

In [ ]:
# ============================================================
# Cell 3: System Dashboard with ipywidgets
# ============================================================
import ipywidgets as widgets
from IPython.display import display, HTML
import psutil
import subprocess

def create_dashboard():
    # RAM
    mem = psutil.virtual_memory()
    ram_bar = widgets.FloatProgress(
        value=mem.percent, min=0, max=100,
        description='RAM:',
        bar_style='info' if mem.percent < 80 else 'warning',
        style={'description_width': '60px', 'bar_color': '#58a6ff'},
        layout=widgets.Layout(width='400px')
    )
    ram_label = widgets.HTML(
        f'<span style="color:#c9d1d9">{mem.used/(1024**3):.1f} / {mem.total/(1024**3):.1f} GB</span>'
    )

    # Disk
    disk = psutil.disk_usage('/')
    disk_bar = widgets.FloatProgress(
        value=disk.percent, min=0, max=100,
        description='Disk:',
        bar_style='info' if disk.percent < 85 else 'danger',
        style={'description_width': '60px', 'bar_color': '#3fb950'},
        layout=widgets.Layout(width='400px')
    )
    disk_label = widgets.HTML(
        f'<span style="color:#c9d1d9">{disk.free/(1024**3):.1f} GB free of {disk.total/(1024**3):.1f} GB</span>'
    )

    # GPU
    gpu_html = '<span style="color:#8b949e">No GPU detected</span>'
    try:
        result = subprocess.run(
            ['nvidia-smi', '--query-gpu=name,utilization.gpu,memory.used,memory.total',
             '--format=csv,noheader'],
            capture_output=True, text=True, timeout=5
        )
        if result.returncode == 0:
            parts = result.stdout.strip().split(',')
            name = parts[0].strip()
            util = parts[1].strip()
            mem_used = parts[2].strip()
            mem_total = parts[3].strip()
            gpu_html = f'<span style="color:#3fb950">🟢 {name} | {util} | {mem_used} / {mem_total}</span>'
    except Exception:
        pass

    gpu_widget = widgets.HTML(gpu_html)

    header = widgets.HTML(
        '<h3 style="color:#58a6ff; margin:0 0 10px 0;">📊 System Resources</h3>'
    )

    dashboard = widgets.VBox([
        header,
        widgets.HBox([ram_bar, ram_label]),
        widgets.HBox([disk_bar, disk_label]),
        widgets.HTML('<b style="color:#c9d1d9">GPU:</b>'),
        gpu_widget,
    ], layout=widgets.Layout(
        padding='15px',
        border='1px solid #30363d',
        border_radius='12px',
        width='600px',
    ))

    display(dashboard)

create_dashboard()


## Cell 4: Khoi Dong Giao Dien Web (QUAN TRONG)

> Sau khi cell nay chay xong, ban se thay **1 URL mau xanh**. Click vao do de mo giao dien.
>
> Tren giao dien: upload video -> chon ngon ngu -> nhan Start -> doi ket qua -> tai video ve.

In [ ]:
# ============================================================
# Cell 4: Khoi dong Streamlit + Cloudflare Tunnel
# ============================================================
# Cell nay se:
#   1. Chay Streamlit server (giao dien web)
#   2. Tao Cloudflare tunnel (de truy cap tu ben ngoai)
#   3. Hien thi URL de ban click vao
# ============================================================

import subprocess
import time
import shutil
import re
from IPython.display import display, HTML

STREAMLIT_PORT = 8501
APP_PATH = 'apps/videotransdub/src/videotransdub/app.py'

# --- 1. Chay Streamlit ---
print('Dang khoi dong giao dien...')
st_proc = subprocess.Popen(
    ['streamlit', 'run', APP_PATH,
     '--server.port', str(STREAMLIT_PORT),
     '--server.headless', 'true',
     '--server.enableCORS', 'false',
     '--server.enableXsrfProtection', 'false',
     '--theme.base', 'dark',
     '--theme.primaryColor', '#58a6ff',
     '--theme.backgroundColor', '#0e1117',
     '--theme.secondaryBackgroundColor', '#161b22',
     '--theme.textColor', '#fafafa',
    ],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
)
time.sleep(5)
print(f'  Streamlit OK (PID: {st_proc.pid})')

# --- 2. Tao Cloudflare tunnel ---
tunnel_url = None
if shutil.which('cloudflared'):
    print('Dang tao duong link truy cap...')
    cf_proc = subprocess.Popen(
        ['cloudflared', 'tunnel', '--url', f'http://localhost:{STREAMLIT_PORT}'],
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
    )
    for _ in range(30):
        time.sleep(1)
        line = cf_proc.stderr.readline().decode('utf-8', errors='ignore')
        match = re.search(r'(https://[a-z0-9-]+\.trycloudflare\.com)', line)
        if match:
            tunnel_url = match.group(1)
            break

# --- 3. Hien thi ket qua ---
if tunnel_url:
    display(HTML(f'''
    <div style="background:#0d3220; border:2px solid #3fb950; border-radius:16px; padding:30px; margin:20px 0; text-align:center;">
        <h1 style="color:#3fb950; margin:0 0 10px 0;">GIAO DIEN DA SAN SANG</h1>
        <p style="color:#c9d1d9; font-size:16px; margin:0 0 20px 0;">Click link ben duoi de mo:</p>
        <a href="{tunnel_url}" target="_blank"
           style="background:#238636; color:white; padding:15px 40px; border-radius:10px;
                  font-size:20px; font-weight:bold; text-decoration:none; display:inline-block;">
            {tunnel_url}
        </a>
        <p style="color:#8b949e; font-size:13px; margin:20px 0 0 0;">
            Mo tren dien thoai, may tinh khac deu duoc. Neu mat ket noi, chay lai cell nay.
        </p>
    </div>
    '''))
else:
    display(HTML('''
    <div style="background:#3d2e00; border:2px solid #d29922; border-radius:16px; padding:30px; margin:20px 0; text-align:center;">
        <h2 style="color:#d29922; margin:0 0 10px 0;">Khong tao duoc link Cloudflare</h2>
        <p style="color:#c9d1d9;">Thu chay lai cell nay. Hoac dung Cell 7 (che do CLI) thay the.</p>
    </div>
    '''))

## Cell 5: Test Pipeline (tuy chon)

> Cell nay tao 1 video test ngan va chay thu pipeline de kiem tra moi thu hoat dong.
> **Khong bat buoc** -- chi dung khi muon kiem tra truoc khi xu ly video that.

In [ ]:
# ============================================================
# Cell 5: Real End-to-End Pipeline Test
# ============================================================
import subprocess
import json
from pathlib import Path

TEST_VIDEO = Path('/content/test_sample.mp4')

# Create a short test video with speech using ffmpeg
if not TEST_VIDEO.exists():
    print('🎬 Creating test video with speech audio...')
    # Generate a 5-second test video with a tone (as placeholder)
    # In real usage, user uploads their own video
    subprocess.run([
        'ffmpeg', '-y',
        '-f', 'lavfi', '-i', 'sine=frequency=440:duration=5',
        '-f', 'lavfi', '-i', 'color=c=black:s=640x360:d=5',
        '-c:v', 'libx264', '-c:a', 'aac',
        '-shortest',
        str(TEST_VIDEO)
    ], capture_output=True)
    print(f'  ✅ Test video: {TEST_VIDEO} ({TEST_VIDEO.stat().st_size:,} bytes)')

# Run the real pipeline with real_free preset
print('\n🚀 Running real pipeline (real_free preset)...')
print('  ASR: faster-whisper (small)')
print('  Translation: Google Free')
print('  TTS: Edge-TTS')
print('  Target: Vietnamese\n')

result = subprocess.run(
    ['python3', '-m', 'videotransdub.cli', 'run',
     '--config', 'apps/videotransdub/configs/default.yaml',
     '--config', 'apps/videotransdub/configs/presets/real_free.yaml',
     '--video-path', str(TEST_VIDEO),
     '--target-language', 'vi',
    ],
    capture_output=True, text=True, timeout=600,
    env={**dict(__import__('os').environ), 'PYTHONPATH': 'apps/videotransdub/src'},
)

print('--- STDOUT ---')
print(result.stdout[-3000:] if len(result.stdout) > 3000 else result.stdout)
if result.returncode != 0:
    print('\n--- STDERR ---')
    print(result.stderr[-2000:] if len(result.stderr) > 2000 else result.stderr)
    print(f'\n❌ Pipeline failed (exit code {result.returncode})')
else:
    print('\n✅ Pipeline completed successfully!')
    # Show output
    workspace_dirs = list(Path('runtime/workspace').iterdir())
    if workspace_dirs:
        ws = sorted(workspace_dirs)[-1]
        manifest = ws / 'manifests' / 'job.json'
        if manifest.exists():
            data = json.loads(manifest.read_text())
            print(f'\n📋 Job: {data.get("job_id")}')
            for stage_name, stage_info in data.get('stages', {}).items():
                print(f'  {stage_name}: {stage_info.get("status", "unknown")}')
            final = data.get('artifacts', {}).get('final_video', '')
            if final and Path(final).exists():
                print(f'\n🎥 Output: {final} ({Path(final).stat().st_size:,} bytes)')


## Cell 6: Sao Chep Video Len Google Drive

> Chay cell nay sau khi pipeline xong de luu video ket qua len Drive (phong mat khi Colab tat).

In [ ]:
# ============================================================
# Cell 6: Sync workspace & output to Google Drive
# ============================================================
import shutil
from pathlib import Path

WORKSPACE_DIR = Path('runtime/workspace')
GDRIVE_VTD = Path('/content/drive/MyDrive/VideoTransDub')

if GDRIVE_VTD.exists():
    print('📁 Syncing to Google Drive...')
    for ws in WORKSPACE_DIR.iterdir():
        if not ws.is_dir():
            continue
        dest = GDRIVE_VTD / 'checkpoints' / ws.name
        # Sync checkpoint and manifest
        manifest_dir = ws / 'manifests'
        if manifest_dir.exists():
            dest_m = dest / 'manifests'
            dest_m.mkdir(parents=True, exist_ok=True)
            for f in manifest_dir.iterdir():
                shutil.copy2(f, dest_m / f.name)

        # Sync output
        output_dir = ws / 'output'
        if output_dir.exists():
            for f in output_dir.iterdir():
                if f.suffix in ('.mp4', '.mkv', '.json'):
                    shutil.copy2(f, GDRIVE_VTD / 'output' / f.name)
                    print(f'  ✅ {f.name} -> Drive')

    print('\n✅ Sync complete!')
    print(f'   Output: {GDRIVE_VTD / "output"}')
    print(f'   Checkpoints: {GDRIVE_VTD / "checkpoints"}')
else:
    print('⚠️ Google Drive not mounted. Run Cell 1 first.')


## Cell 7: Che Do Dong Lenh (thay the cho giao dien)

> Neu giao dien web bi loi hoac ban thich dung dong lenh, sua cac bien ben duoi roi chay cell nay.

In [ ]:
# ============================================================
# Cell 7: Direct CLI execution (alternative to UI)
# ============================================================

# --- Configure these ---
VIDEO_PATH = '/content/drive/MyDrive/your_video.mp4'  # Change this!
PRESET = 'real_free'  # Options: real_free, fast_free, qwen_free, balanced, quality_api, mock
TARGET_LANG = 'vi'
SOURCE_LANG = 'auto'

# For qwen_free preset, set your API key:
import os
os.environ['QWEN_API_KEY'] = 'sk-abd0a732837b414696fc23a8f933aa3b'

# --- Run ---
!PYTHONPATH=apps/videotransdub/src python3 -m videotransdub.cli run \
    --config apps/videotransdub/configs/default.yaml \
    --config apps/videotransdub/configs/presets/{PRESET}.yaml \
    --video-path "{VIDEO_PATH}" \
    --target-language {TARGET_LANG} \
    --source-language {SOURCE_LANG}